# 01 - DRC Gross Jump-to-Default

Synthetic engineering and validation evidence, not final regulatory capital.

Verify gross JTD mechanics for synthetic default-risk positions after showing the public capital entrypoint.

Use this notebook as a companion to [`PACKAGE_JOURNEY.md`](../docs/PACKAGE_JOURNEY.md) and [`examples/run_demo.py`](../examples/run_demo.py). The happy path is the public package API: construct DRC positions and a calculation context, then call `frtb_drc.calculate_drc_capital`.

## Raw inputs your upstream risk engine must emit

A DRC upstream engine should emit one row per default-risk position with stable position and source-row identifiers, desk/legal-entity scope, risk class, instrument type, issuer or tranche/index identifiers where applicable, bucket key, seniority or credit quality, notional or market value, maturity, currency, lineage fields, and citation identifiers. The non-securitisation table contract is documented in [`docs/schemas/input_table/frtb_drc.nonsec.schema.json`](../../../docs/schemas/input_table/frtb_drc.nonsec.schema.json); package-specific context and fixture notes live in [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md).


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from examples.drc_nonsec_fixture import load_drc_nonsec_v2_fixture, run_fixture_workflow
from frtb_drc.demo_data import ALL_POSITIONS, DEMO_CONTEXT

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def markdown_table(headers: list[str], rows: list[list[str]]) -> Markdown:
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return Markdown("\n".join([header, separator, *body]))


fixture = load_drc_nonsec_v2_fixture()
positions = fixture.positions
context = fixture.context
expected = fixture.expected_outputs
display(
    Markdown(
        f"Loaded fixture `{expected['fixture_id']}`: "
        f"`{len(positions)}` positions, "
        f"profile `{context.profile_id}`, "
        f"as-of `{context.calculation_date}`."
    )
)

## Public API happy path

The primary user contract is public API first: pass validated `DrcPosition` records and `DrcCalculationContext` to `calculate_drc_capital`. The implementation-detail sections below inspect intermediate mechanics for validation and auditability.


In [ ]:
from frtb_drc import calculate_drc_capital

public_result = calculate_drc_capital(positions, context=context)
print(f"Total DRC capital: {public_result.total_drc:,.2f}")
print(f"Input hash       : {public_result.input_hash}")
print(f"Profile hash     : {public_result.profile_hash}")


## Implementation details / math verification - Gross JTD mechanics


In [ ]:
from frtb_drc.gross_jtd import calculate_gross_jtd
from frtb_drc.reference_data import US_NPR_2_0_PROFILE_ID

results = [calculate_gross_jtd(p, profile_id=US_NPR_2_0_PROFILE_ID) for p in positions]

rows = [
    [
        g.position_id,
        g.bucket_key,
        p.default_direction,
        p.seniority,
        f"{p.notional:>12,.0f}",
        f"{g.lgd_rate:.0%}",
        f"{p.cumulative_pnl or 0:>12,.0f}",
        f"{g.gross_jtd:>12,.2f}",
    ]
    for g, p in zip(results, positions)
    if g.gross_jtd > 0 or p.seniority in ("NOT_RECOVERY_LINKED",)
]
display(markdown_table(
    ["Position", "Bucket", "Dir", "Seniority", "Notional", "LGD", "PnL", "Gross JTD"],
    rows,
))

## Special cases

### 1. NOT\_RECOVERY\_LINKED (LGD = 0)

`theta-pharma` holds a NOT\_RECOVERY\_LINKED instrument.  LGD = 0 → gross JTD = 0
regardless of notional.  The position contributes nothing to capital.

In [ ]:
theta = next(g for g in results if "theta" in g.position_id)
print(f"theta-pharma: notional={theta.notional:,.0f}  lgd={theta.lgd_rate:.0%}  gross_jtd={theta.gross_jtd:.2f}")

### 2. Large negative P&L floors gross JTD to zero

`mu-industries` holds LONG SENIOR\_DEBT with a large unrealised loss
(`cumulative_pnl = -400 000`).

```
raw_jtd = 0.75 * 500 000 + (-400 000) = 375 000 - 400 000 = -25 000
gross_jtd = max(-25 000, 0) = 0
```

In [ ]:
mu_pos = next(p for p in positions if "mu" in p.position_id)
mu_g = next(g for g in results if "mu" in g.position_id)
print(
    f"mu-industries: notional={mu_pos.notional:,.0f}  lgd={mu_g.lgd_rate:.0%}"
    f"  pnl={mu_pos.cumulative_pnl:,.0f}  gross_jtd={mu_g.gross_jtd:.2f}"
)

### 3. Positive P&L on a LONG position increases gross JTD

`delta-retail` has `cumulative_pnl = +80 000` (unrealised gain).
The bond is trading above par; the additional mark-to-market exposure is at risk on default.

```
raw_jtd = 1.0 * 600 000 + 80 000 = 680 000
```

In [ ]:
delta_pos = next(p for p in positions if "delta" in p.position_id)
delta_g = next(g for g in results if "delta" in g.position_id)
print(
    f"delta-retail: notional={delta_pos.notional:,.0f}  lgd={delta_g.lgd_rate:.0%}"
    f"  pnl={delta_pos.cumulative_pnl:,.0f}  gross_jtd={delta_g.gross_jtd:.2f}"
)

## Gross JTD chart — long positions by bucket

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

buckets = ["CORPORATE", "NON_US_SOVEREIGN", "PSE_GSE", "DEFAULTED"]
colours = ["#2563EB", "#059669", "#D97706", "#DC2626"]

fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=False)
for ax, bucket, colour in zip(axes, buckets, colours):
    bucket_results = [
        (g.position_id.split("-", 2)[-1][:18], g.gross_jtd)
        for g, p in zip(results, positions)
        if p.bucket_key == bucket and p.default_direction == "LONG" and g.gross_jtd > 0
    ]
    if not bucket_results:
        ax.set_visible(False)
        continue
    labels, values = zip(*bucket_results)
    y = np.arange(len(labels))
    ax.barh(y, [v / 1e6 for v in values], color=colour, alpha=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel("Gross JTD (USD M)")
    ax.set_title(bucket, fontsize=9)

fig.suptitle("Gross JTD — long positions by bucket", fontsize=11)
fig.tight_layout()
plt.show()

## See also

- [`packages/frtb-drc/docs/PACKAGE_JOURNEY.md`](../docs/PACKAGE_JOURNEY.md) for the package workflow and maturity path.
- [`packages/frtb-drc/docs/DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md) for fixture and input-shape notes.
- [`docs/CLIENT_INTEGRATION.md`](../../../docs/CLIENT_INTEGRATION.md) for suite-level client integration guidance.
- Sibling component journeys: [`frtb-sbm`](../../frtb-sbm/docs/PACKAGE_JOURNEY.md), [`frtb-rrao`](../../frtb-rrao/docs/PACKAGE_JOURNEY.md), [`frtb-cva`](../../frtb-cva/docs/PACKAGE_JOURNEY.md), [`frtb-ima`](../../frtb-ima/docs/PACKAGE_JOURNEY.md), and [`frtb-orchestration`](../../frtb-orchestration/docs/PACKAGE_JOURNEY.md).
